# Silver Layer — Order Payments Transformation

Transforms `bronze.order_payments` into `silver.order_payments`. Filters nulls, deduplicates by composite key `(order_id, payment_sequential)`, cleans payment_type, and validates with four DQ checks.

Rows violating the value-range rules (`payment_value > 0`, `payment_installments > 0`) are quarantined to `quarantine.order_payments` with a `violation_reasons` column. Clean rows go to Silver.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%run ../Utilities/silver_dq_checks

# Silver Layer — Data Quality Checks

This notebook defines reusable data quality check functions used across the Silver layer transformation notebooks.

Each function validates a specific property of a transformed DataFrame before it is written to a Silver Delta table. On success the function prints a PASSED message; on failure it raises a `ValueError` with details, stopping the pipeline.

## Checks Defined Here

* `check_not_null` — verifies that specified columns have no null or blank values
* `check_unique` — verifies that the given key (single or composite) has no duplicates
* `check_row_count` — verifies that the Silver row count falls within an expected percentage range of the Bronze source

## How To Use

Import these functions into a Silver transformation notebook by running:
​```
%run ../Utilities/silver_dq_checks
​```
Then call the functions on the transformed DataFrame before writing to Silver.

### check_not_null
Validates that the specified columns contain no null or blank/whitespace values.

### check_unique
Validates that the specified key columns produce unique rows. Supports single-column or composite keys.

### check_row_count
Validates that the Silver row count is within an acceptable percentage range of the Bronze source row count.


### check_event_sequence
Validates that timestamp columns appear in the expected chronological order. Skips rows where either timestamp in a pair is null. Raises on any violation.

### identify_event_sequence_violations
Adds one boolean column per consecutive timestamp pair to mark whether each row violates the expected order. Returns the annotated DataFrame without raising — callers use it to split clean rows from violations.

### check_value_range
Validates that values in specified columns meet a minimum threshold rule (e.g. `price > 0`, `freight_value >= 0`). Each rule is a dict specifying column, min_value, and inclusivity. Raises on any violation, reporting all rules that failed.

In [0]:
catalog_name = 'retail_lakehouse'
bronze_schema = 'bronze'
silver_schema = 'silver'
quarantine_schema = 'quarantine'

### Transformations

In [0]:
df_payments = spark.read.table(f'{catalog_name}.{bronze_schema}.order_payments')
transformed_payments_df  = df_payments.filter((col('order_id').isNotNull()) & (col('payment_sequential').isNotNull())
                                      & (col('payment_type').isNotNull()) 
                                      & (col('payment_installments').isNotNull()) 
                                      & (col('payment_value').isNotNull()))\
                                          .dropDuplicates(['order_id','payment_sequential'])\
                                              .withColumn('payment_type',lower(trim(col('payment_type'))))\
                                                  .withColumn('silver_processed_at',current_timestamp())


### Identify invalid payment_value and payment_installments column values
Validates the payment_value and payment_sequential columns which have values less than or equal to zero. 
Split the violated and good records into different dataframe by using filtering.

In [0]:
df_violations = transformed_payments_df.filter((col('payment_installments') <= 0) | (col('payment_value') <= 0))

df_clean = transformed_payments_df.filter((col('payment_installments') > 0) & (col('payment_value') > 0))


df_quarantine = df_violations.withColumn(
    'violation_reasons', 
    when((col('payment_installments') <= 0) & (col('payment_value') <= 0),
         lit('payment_installments <= 0 and payment_value <= 0'))
    .when(col('payment_installments') <= 0,
          lit('payment_installments <= 0'))
    .when(col('payment_value') <= 0,
          lit('payment_value <= 0'))
)


## Writing quarantine dataframe to quarantine schema


In [0]:
df_quarantine.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{quarantine_schema}.order_payments')


### Data quality checks

In [0]:
check_not_null(df_clean,['order_id','payment_value','payment_sequential','payment_type','payment_installments'])
check_unique(df_clean,['order_id','payment_sequential'])
check_row_count(df_clean,f'{catalog_name}.{bronze_schema}.order_payments',85,100)
check_value_range(df_clean,[{'column':'payment_value','min_value':0,'inclusive':False},{'column':'payment_installments','min_value':0,'inclusive':False}])

check_not_null: PASSED
check_unique: PASSED
check_row_count: PASSED
check_value_range: PASSED


## Writing clean dataframe to silver schema

In [0]:
df_clean.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{silver_schema}.order_payments')
